# Summarization

Can a machine summarize a document?

In [1]:
from google.colab import drive
drive.mount('/content/drive')  # Add My Drive/<>

import os
os.chdir('drive/My Drive')
os.chdir('Books_Writings/NLPBook/')

Mounted at /content/drive


In [2]:
%%capture
%pylab inline
import pandas as pd
import os
%load_ext rpy2.ipython
import textwrap

## Types of Summarization

There are two broad types of text summarization:

1. Extractive: provide the most meaningful extracted subsample from the text.
2. Abstractive: generate new language that explains the document more briefly.

There are some metrics for the quality of summarization, see: http://nlpprogress.com/english/summarization.html

But now we have "Generative" summarization using LLMs. Ask yourself when this is better and when it is worse.

## Jaccard Summarizer

Here we present a simple approach to extractive summarization.

A document $D$  is comprised of  $m$  sentences  $s_i,i=1,2,...,m$, where each  $s_i$  is a set of words. We compute the pairwise overlap between sentences using the Jaccard similarity index:

$$
J_{ij} = J(s_i,s_j)=\frac{|s_i \cap s_j|}{|s_i \cup s_j|} = J_{ji}
$$

The overlap is the ratio of the size of the intersect of the two word sets in sentences  $s_i$  and  $s_j$, divided by the size of the union of the two sets. The similarity score of each sentence is computed as the row sums of the Jaccard similarity matrix.

$$
S_i=\sum_{j=1}^m J_{ij}
$$

## Generating the summary

Once the row sums are obtained, they are sorted and the summary is the first  $n$  sentences based on the  $S_i$  values.

In [3]:
%%R
# FUNCTION TO RETURN n SENTENCE SUMMARY
# Input: array of sentences (text)
# Output: n most common intersecting sentences
text_summary = function(text, n) {
  m = length(text)  # No of sentences in input
  jaccard = matrix(0,m,m)  #Store match index
  for (i in 1:m) {
    for (j in i:m) {
      a = text[i]; aa = unlist(strsplit(a," "))
      b = text[j]; bb = unlist(strsplit(b," "))
      jaccard[i,j] = length(intersect(aa,bb))/
                          length(union(aa,bb))
      jaccard[j,i] = jaccard[i,j]
    }
  }
  similarity_score = rowSums(jaccard)
  res = sort(similarity_score, index.return=TRUE,
          decreasing=TRUE)
  idx = res$ix[1:n]
  summary = text[idx]
}

## One Function to Rule All Text in R

Also, a quick introduction to the tm package in R: https://cran.r-project.org/web/packages/tm/vignettes/tm.pdf

Install (if needed from the command line): `conda install -c r r-tm` or install it as shown below.

In [4]:
%%R
install.packages("tm", quiet=TRUE)
# ! conda install -c conda-forge r-tm -y
# ! conda install -c r r-tm -y

also installing the dependencies ‘NLP’, ‘slam’, ‘BH’



In [5]:
%%R
library(tm)
library(stringr)
#READ IN TEXT FOR ANALYSIS, PUT IT IN A CORPUS, OR ARRAY, OR FLAT STRING
#cstem=1, if stemming needed
#cstop=1, if stopwords to be removed
#ccase=1 for lower case, ccase=2 for upper case
#cpunc=1, if punctuation to be removed
#cflat=1 for flat text wanted, cflat=2 if text array, else returns corpus
read_web_page = function(url,cstem=0,cstop=0,ccase=0,cpunc=0,cflat=0) {
    text = readLines(url)
    text = text[setdiff(seq(1,length(text)),grep("<",text))]
    text = text[setdiff(seq(1,length(text)),grep(">",text))]
    text = text[setdiff(seq(1,length(text)),grep("]",text))]
    text = text[setdiff(seq(1,length(text)),grep("}",text))]
    text = text[setdiff(seq(1,length(text)),grep("_",text))]
    text = text[setdiff(seq(1,length(text)),grep("\\/",text))]
    ctext = Corpus(VectorSource(text))
    if (cstem==1) { ctext = tm_map(ctext, stemDocument) }
    if (cstop==1) { ctext = tm_map(ctext, removeWords, stopwords("english"))}
    if (cpunc==1) { ctext = tm_map(ctext, removePunctuation) }
    if (ccase==1) { ctext = tm_map(ctext, tolower) }
    if (ccase==2) { ctext = tm_map(ctext, toupper) }
    text = ctext
    #CONVERT FROM CORPUS IF NEEDED
    if (cflat>0) {
        text = NULL
        for (j in 1:length(ctext)) {
            temp = ctext[[j]]$content
            if (temp!="") { text = c(text,temp) }
        }
        text = as.array(text)
    }
    if (cflat==1) {
        text = paste(text,collapse="\n")
        text = str_replace_all(text, "[\r\n]" , " ")
    }
    result = text
}

Loading required package: NLP


## Example: Summarization

We will use a sample of text that I took from Bloomberg news. It is about the need for data scientists.

In [6]:
%%R
url = "NLP_data/dstext_sample.txt"   #You can put any text file or URL here
text = read_web_page(url,cstem=0,cstop=0,ccase=0,cpunc=0,cflat=1)
print(length(text[[1]]))

[1] 1


In [7]:
text = %Rget text
text = text[0]
print(textwrap.fill(text, width=80))

THERE HAVE BEEN murmurings that we are now in the “trough of disillusionment” of
big data, the hype around it having surpassed the reality of what it can
deliver.  Gartner suggested that the “gravitational pull of big data is now so
strong that even people who haven’t a clue as to what it’s all about report that
they’re running big data projects.”  Indeed, their research with business
decision makers suggests that organisations are struggling to get value from big
data. Data scientists were meant to be the answer to this issue. Indeed, Hal
Varian, Chief Economist at Google famously joked that “The sexy job in the next
10 years will be statisticians.” He was clearly right as we are now used to
hearing that data scientists are the key to unlocking the value of big data.
This has created a huge market for people with these skills. US recruitment
agency, Glassdoor, report that the average salary for a data scientist is
$118,709 versus $64,537 for a skilled programmer. And a McKinsey study 

In [8]:
%%R
text2 = strsplit(text,". ",fixed=TRUE)  #Special handling of the period.
text2 = text2[[1]]
print(text2)

 [1] "THERE HAVE BEEN murmurings that we are now in the “trough of disillusionment” of big data, the hype around it having surpassed the reality of what it can deliver"                                                                                                                                                     
 [2] " Gartner suggested that the “gravitational pull of big data is now so strong that even people who haven’t a clue as to what it’s all about report that they’re running big data projects.”  Indeed, their research with business decision makers suggests that organisations are struggling to get value from big data"
 [3] "Data scientists were meant to be the answer to this issue"                                                                                                                                                                                                                                                             
 [4] "Indeed, Hal Varian, Chief Economist at G

In [9]:
%%R
res = text_summary(text2,5)
print(res)

[1] " Gartner suggested that the “gravitational pull of big data is now so strong that even people who haven’t a clue as to what it’s all about report that they’re running big data projects.”  Indeed, their research with business decision makers suggests that organisations are struggling to get value from big data"
[2] "The focus on the data scientist often implies a centralized approach to analytics and decision making; we implicitly assume that a small team of highly skilled individuals can meet the needs of the organisation as a whole"                                                                                         
[3] "May be we are investing too much in a relatively small number of individuals rather than thinking about how we can design organisations to help us get the most from data assets"                                                                                                                                      
[4] "The problem with a centralized ‘IT-style’ ap

## Text Summarization with Python

This is a approach that distills a document down to its most important sentences. The idea is very simple. The algorithm simply focuses on the essence of a document. The customer use case is that the quantity of reading is too high and a smaller pithy version would be great to have.

However, in the absence of an article/document, I have some examples where we download an article using selector gadget, Beautiful Soup, and extract the text of the article. But the summarizer/compressor assumes that the article is clean flat file text.

https://www.dataquest.io/blog/web-scraping-tutorial-python/

Install these if needed:

In [10]:
!pip install lxml
!pip install cssselect
!pip install nltk

In [34]:
# Read in the news article from the URL and extract only the title and text of the article.
# Some examples provided below.

import requests
from lxml.html import fromstring
url = "https://www.allthingsdistributed.com/2025/10/better-with-age.html"
# url = "https://www.theverge.com/2023/10/4/23903986/sam-bankman-fried-opening-statements-trial-fraud"
# url = "https://www.nytimes.com/2023/10/03/us/politics/kevin-mccarthy-speaker.html"
# url = "https://www.theatlantic.com/technology/archive/2022/04/doxxing-meaning-libs-of-tiktok/629643/"
# url = 'https://economictimes.indiatimes.com/news/economy/policy/a-tax-cut-for-you-in-budget-wont-give-india-the-boost-it-needs/articleshow/73476138.cms?utm_source=Colombia&utm_medium=C1&utm_campaign=CTN_ET_hp&utm_content=18'


In [37]:
html = requests.get(url, timeout=10).text

#See: http://infohost.nmt.edu/~shipman/soft/pylxml/web/etree-fromstring.html
doc = fromstring(html)

#http://lxml.de/cssselect.html#the-cssselect-method
doc.cssselect("section") # all things
# doc.cssselect(".lg\:max-w-none") # verge
# doc.cssselect(".evys1bk0") # nytimes
# doc.cssselect(".Normal")  #economic times
# doc.cssselect(".ArticleParagraph_root__wy3UI")   #Atlantic

[<Element section at 0x7906c8f83ac0>]

In [48]:
#economic times
# x = doc.cssselect(".Normal")
# news = x[0].text_content()
# print(news)

# Verge
# x = doc.cssselect(".lg\:max-w-none")

#nytimes
# x = doc.cssselect(".StoryBodyCompanionColumn")

# Atlantic
# x = doc.cssselect(".ArticleParagraph_root__wy3UI")

# All things
x = doc.cssselect("section")

news = " ".join([x[j].text_content() for j in range(len(x))])
news

'Development gets better with AgeOctober 01, 2025 • 856 wordsHe has heard the whispers, â\x80\x9che is getting older, who will replace him?â\x80\x9d People asking him with a straight face, â\x80\x9cwhen will you retire?â\x80\x9d After close to 25 years at Amazon, where each year has been different and amazing, He feels as young as the day he decided to leave academia and join Amazon.The thing about getting older as a developer, is that you have seen a lot and encountered many of the problems younger developers are facing (even if they look a little different on first glance). If youâ\x80\x99ve been around the block as many times as some of us have, youâ\x80\x99ll have earned battle scars along the way. There are days in war rooms you will never forget. You have experimented a lot, and you have failed more times than you care to remember. You have half-a-head full of what is practical and works. And a quarter of that space has been trained to look for red flags, scanning for things that

Make sure the text you extracted is in string form. Then convert the article into individual sentences. Put the individual sentences into a list. Use BeautifulSoup for this.

In [40]:
from bs4 import BeautifulSoup
news = BeautifulSoup(news,'lxml').get_text()
print(textwrap.fill(news, width=80))
type(news)

Development gets better with AgeOctober 01, 2025 • 856 wordsHe has heard the
whispers, âhe is getting older, who will replace him?â People asking him
with a straight face, âwhen will you retire?â After close to 25 years at
Amazon, where each year has been different and amazing, He feels as young as the
day he decided to leave academia and join Amazon.The thing about getting older
as a developer, is that you have seen a lot and encountered many of the problems
younger developers are facing (even if they look a little different on first
glance). If youâve been around the block as many times as some of us have,
youâll have earned battle scars along the way. There are days in war rooms you
will never forget. You have experimented a lot, and you have failed more times
than you care to remember. You have half-a-head full of what is practical and
works. And a quarter of that space has been trained to look for red flags,
scanning for things that you know will go wrong.Whatâs left

str

In [41]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
from nltk.tokenize import sent_tokenize   # To get separate sentences
sentences = sent_tokenize(news)
print("Number of sentences =", len(sentences))
for s in sentences:
    print(textwrap.fill(s, width=80), end="\n\n")

Number of sentences = 40
Development gets better with AgeOctober 01, 2025 • 856 wordsHe has heard the
whispers, âhe is getting older, who will replace him?â People asking him
with a straight face, âwhen will you retire?â After close to 25 years at
Amazon, where each year has been different and amazing, He feels as young as the
day he decided to leave academia and join Amazon.The thing about getting older
as a developer, is that you have seen a lot and encountered many of the problems
younger developers are facing (even if they look a little different on first
glance).

If youâve been around the block as many times as some of us have, youâll
have earned battle scars along the way.

There are days in war rooms you will never forget.

You have experimented a lot, and you have failed more times than you care to
remember.

You have half-a-head full of what is practical and works.

And a quarter of that space has been trained to look for red flags, scanning for
things that you kn

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [42]:
# Python Summarizer
import re
# Pass in a list of sentences, returns a n sentence summary
def text_summarizer(sentences, n_summary):
    n = len(sentences)
    x = [re.split('[ ,.]',j) for j in sentences]
    jaccsim = array(zeros(n*n)).reshape((n,n))
    for i in range(n):
        for j in range(i,n):
            jaccsim[i,j] = len(set(x[i]).intersection(set(x[j])))/len(set(x[i]).union(set(x[j])))
            jaccsim[j,i] = jaccsim[i,j]
    #Summary
    idx = argsort(sum(jaccsim, axis=0))[::-1][:n_summary]  #reverse sort
    summary = [sentences[j] for j in list(idx)]
    #Anomalies
    idx = argsort(sum(jaccsim, axis=0))[:n_summary]
    anomalies = [sentences[j] for j in list(idx)]
    return summary, anomalies

In [43]:
# Get the summary and the anomaly sentences
summary, anomalies = text_summarizer(sentences, int(len(sentences)/4))
summ = "  ".join(summary)
print(textwrap.fill(summ, width=80))

Itâs the best part of our job.  In the hands of a seasoned builder with a
healthy dose of scepticism, it is powerful.  That they have very strong feelings
of FOMO (the fear of missing out).And the older developer knows that this is
exactly the time to press the pause button.  And a quarter of that space has
been trained to look for red flags, scanning for things that you know will go
wrong.Whatâs left in your head is used for creativity.  The magic was just let
out of the bottle, and because it was so unexpected, the hype absolutely
exploded.  Let that sink in for a second.  You have experimented a lot, and you
have failed more times than you care to remember.  And sometimes, the solution
will be generative AI.But as an older developer, you already knew this.Now, go
build!  You have half-a-head full of what is practical and works.  That’s the
value of having seen patterns repeat over decades - you know which ones work.The
older developer isnât worried about the barrage of new mod

In [44]:
for a in anomalies:
    print(a)

Who else gets to do that?
Have an in-depth conversation with your customer, listen, dive deep into their challenges, suggest architectures, migrations, and tools.
We went back to our roots, democratizing access to technology (models in this case), giving customers choice, keeping privacy and security as our top priorities, providing the guardrails companies need for safety and compliance, and leveraging automated reasoning to reduce potential model errors.
There are days in war rooms you will never forget.
Heâs also read Marc Brookerâs fantastic article about LLM-driven development, and will probably follow his advice.Almost every customer I speak with asks: âWhat should we be doing with gen AI?â The best response Iâve seen so far is from Byron Cook, one of our brilliant scientists: âSorry for not answering your question immediately, but why did you ask me this question?âYouâll find that 90% of the answers you get back are not because they think generative AI will solve

## Modern Methods

- Extractive Summarization vs Abstractive Summarization

- Summarization with pointer networks: https://drive.google.com/file/d/1fAgr85WAQU8OXYkwifuF4Ep-LXfrwinv/view?usp=sharing

- Use Hugging Face Transformers as shown next: https://huggingface.co/transformers/main_classes/pipelines.html

In [45]:
!pip install transformers

In [46]:
from transformers import pipeline
summarizer = pipeline("summarization")

No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


In [49]:
# All in one example
html = requests.get(url, timeout=10).text
doc = fromstring(html)
# x = doc.cssselect(".ArticleParagraph_root__wy3UI")
# x = doc.cssselect(".lg\:max-w-none")
news = " ".join([x[j].text_content() for j in range(len(x))])
news = BeautifulSoup(news,'lxml').get_text()
print(len(news))
if len(news)>1024:   # max seq length
    news = news[:1024]
summ = summarizer(news, max_length=int(len(news)/4), min_length=25)
print(summ)

5145
[{'summary_text': ' Development gets better with age, says senior Amazon developer . He has heard the whispers, â\x80\x9d he is getting older, who will replace him?'}]


Try this additional blog post for more on the T5 (text to text transfer transformer) summarizer.

https://towardsdatascience.com/simple-abstractive-text-summarization-with-pretrained-t5-text-to-text-transfer-transformer-10f6d602c426

This is a nice web site explaining Hugging Face transformers: https://zenodo.org/record/3733180#.X40RxEJKjlx

And the paper: https://arxiv.org/pdf/1910.10683.pdf

And here is a nice application of the same: https://towardsdatascience.com/summarization-has-gotten-commoditized-thanks-to-bert-9bb73f2d6922

## Long document summarization

This is not feasible unless we break up the text into maximal chunk sizes and do the summary piecemeal.

In [50]:
html = requests.get(url, timeout=10).text
doc = fromstring(html)
# x = doc.cssselect(".ArticleParagraph_root__wy3UI")
# x = doc.cssselect(".lg\:max-w-none")
news = " ".join([x[j].text_content() for j in range(len(x))])
news = BeautifulSoup(news,'lxml').get_text()
print("Size of article =",len(news)," | #Chunks =",int(len(news)/1024))
for j in range(0,len(news),1024):
    print(summarizer(news[j:j+1024], max_length=int(len(news)/4), min_length=25))

Your max_length is set to 1286, but your input_length is only 256. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=128)


Size of article = 5145  | #Chunks = 5


Your max_length is set to 1286, but your input_length is only 266. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=133)


[{'summary_text': ' Development gets better with age, says senior Amazon developer . He has heard the whispers, â\x80\x9d he is getting older, who will replace him?'}]


Your max_length is set to 1286, but your input_length is only 231. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=115)


...

[{'summary_text': ' The magic was just let out of the bottle, and because it was so unexpected, the hype exploded . This feels strange to us, because weâ\x80\x99ve been used to seeing our software evolve with minor version bumps that take a year or more to come out .'}]


Your max_length is set to 1286, but your input_length is only 270. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=135)


[{'summary_text': ' The older developer isnâ\x80\x99�s worried about the barrage of new model announcements and feature releases that come out every week . The value of having seen patterns repeat over decades - you know which ones work .'}]


Your max_length is set to 1286, but your input_length is only 11. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=5)


[{'summary_text': ' An older developer knows this is exactly the time to press the pause button . 90% of the answers you get back are not because they think generative AI will solve a specific problem that their business is encountering, but because they’re anxious .'}]
[{'summary_text': " Now, go build!  knew this. Now, you're going to be able to build your dream home ."}]
